

# Recipe generator

## Imports <a name="im"></a>

In [54]:
import os
import re
import sys
from collections import Counter, defaultdict
from urllib.request import urlopen
from hashlib import sha1

import numpy as np
import numpy.random as npr
import pandas as pd

In [55]:
import string

punc = string.punctuation + "”" + "’"
punc

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~”’'

In [56]:
class MarkovModelWords:
    def __init__(self, n):
        """
        Initialize a word-level Markov model.

        Parameters
        ----------
        n : int
            Number of previous words to use as context.
        """
        self.n = n
        self.probabilities_ = None
        self.frequencies_ = None
        self.starting_words = None

    def fit(self, tokens):
        """
        Fit a word-level Markov model.

        Parameters
        ----------
        tokens : tuple or list of str
            A tokenized text corpus, e.g., ("peanut", "butter", "and", "jelly").
        """

        # Store the first n words as the starting context for generation
        self.starting_words = tuple(tokens[: self.n])

        # Make the token sequence circular so the model does not get stuck
        circular_tokens = tuple(tokens) + tuple(tokens[: self.n])

        # Step 1: Count how often each word follows each n-word context
        frequencies = defaultdict(Counter)

        for i in range(len(tokens)):
            context = circular_tokens[i : i + self.n]
            next_word = circular_tokens[i + self.n]
            frequencies[context][next_word] += 1.0

        # Step 2: Convert counts into probabilities
        self.probabilities_ = defaultdict(dict)

        for context, counts in frequencies.items():
            self.probabilities_[context]["words"] = list(counts.keys())

            probs = np.array(list(counts.values()))
            probs = probs / np.sum(probs)

            self.probabilities_[context]["probs"] = probs

        self.frequencies_ = frequencies

    def generate(self, seq_len, seed=None):
        """
        Generate a sequence of words.

        Parameters
        ----------
        seq_len : int
            Number of words to generate.

        seed : str, list[str], tuple[str], optional
            Initial context to start generation. If None, the first
            n words from the training corpus are used.

        Returns
        -------
        str
            Generated text.
        """
        n = self.n

        if seed is None:
            sequence = list(self.starting_words)
        else:
            if isinstance(seed, str):
                seed = word_tokenize(seed)

            if len(seed) < n:
                raise ValueError(
                    f"Seed must contain at least {n} words."
                )

            sequence = list(seed)

        while len(sequence) < seq_len:
            context = tuple(sequence[-n:])

            # Stop if we've never seen this context
            if context not in self.probabilities_:
                break

            probs = self.probabilities_[context]
            next_word = npr.choice(probs["words"], p=probs["probs"])
            sequence.append(next_word)

        output = sequence[0]
        for word in sequence[1:]:
            if word == "\n":
                output += "\n"
            elif word in punc:
                output += word
            elif output.endswith("\n"):
                output += word
            else:
                output += " " + word

        return output

Let's test our code by getting some novel recipe names. Let's bring back the recipe names corpus we used in DSCI 563.

Download [`RAW_recipes.csv`](https://www.kaggle.com/shuyangli94/food-com-recipes-and-user-interactions?select=RAW_recipes.csv) and place it in the `data` directory of your lab folder. When you open the link, you'll see multiple datasets. You only need to download `RAW_recipes.csv`, which you can select from the "Data Explorer" section. As usual, do not push the CSV file to your repository.

In [ ]:
orig_recipes_df = pd.read_csv("data/RAW_recipes.csv")
orig_recipes_df = orig_recipes_df.dropna()
text = "\n".join(orig_recipes_df["name"].tolist())

In [ ]:
from nltk.tokenize import word_tokenize

tokens = []

for recipe in orig_recipes_df["name"]:
    tokens.extend(word_tokenize(recipe))
    tokens.append("\n")          # Preserve recipe boundary

text_tok = tuple(tokens)

In [ ]:
orig_recipes_df['name'].head(50)

In [ ]:
# BEGIN SOLUTION
np.random.seed(8)
for n in np.arange(1, 5):
    print("CONTEXT LENGTH = %d words\n" %n)
    model = MarkovModelWords(n=n)
    model.fit(text_tok)
    print(model.generate(40, seed="chocolate vanilla ice cream"))
    print("\n") 
# END SOLUTION    

<br><br>